# Research Paper RAG Assistant (V2.1)

A custom Retrieval-Augmented Generation (RAG) system for answering questions from academic research papers across both single-column and IEEE two-column formats.

## Project Architecture

PDF
->
PyMuPDF Extraction
->
Metadata / Content Split
->
Section Detection
->
Section-aware Chunking
->
Embeddings (Sentence Transformers)
->
FAISS Retrieval
->
Query Routing
->
Local LLM (Qwen via Ollama)
->
Answer Generation

STEP 1: EXTRACTION TEXT USING PYMUPDF

In [ ]:
!pip install PyMuPDF
import fitz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 71.0 MB/s eta 0:00:00


In [ ]:
import fitz


def extract_text_pymupdf(pdf_path):
    """
    Extract text from PDF using PyMuPDF line reconstruction.

    Returns:
        str: Full document text
    """

    doc = fitz.open(pdf_path)

    document_lines = []

    for page_num, page in enumerate(doc, start=1):

        page_lines = []

        page_dict = page.get_text("dict")

        for block in page_dict.get("blocks", []):

            if "lines" not in block:
                continue

            for line in block["lines"]:

                line_text = " ".join(
                    span["text"].strip()
                    for span in line["spans"]
                    if span["text"].strip()
                )

                if line_text:
                    page_lines.append(line_text)

        document_lines.extend(page_lines)

    doc.close()

    return "\n".join(document_lines)

STEP 1.1 Extracting section hadings from the extracted text

In [ ]:
KNOWN_SECTIONS = [
    "abstract",
    "introduction",
    "related work",
    "background",
    "literature review",
    "literature survey",
    "proposed model",
    "proposed architecture",
    "background",
    "objectives",
    "methodology",
    "methods",
    "materials and methods",
    "experimental setup",
    "experiments",
    "results",
    "results and discussion",
    "experimental results",
    "discussion",
    "conclusion",
    "future work",
    "references"
]

In [ ]:
import re


def extract_section_headings(full_text):

    headings = []

    lines = full_text.split("\n")

    for line in lines:

        cleaned = line.strip()

        if not cleaned:
            continue

        lower = cleaned.lower()

        # Direct match
        if lower in KNOWN_SECTIONS:

            headings.append(cleaned)

            continue

        # IEEE patterns
        patterns = [
            r"^[IVXLC]+\.\s+(.+)$",      # I. INTRODUCTION
            r"^\d+\.\s+(.+)$",           # 1. Introduction
            r"^\d+\s+(.+)$"             # 1 Introduction
        ]

        for pattern in patterns:

            match = re.match(
                pattern,
                cleaned,
                flags=re.IGNORECASE
            )

            if match:

                section_name = match.group(1).strip()

                if section_name.lower() in KNOWN_SECTIONS:

                    headings.append(section_name)

                break

    return headings

STEP 2: Separating metadata and content sections

In [ ]:
import re


def split_metadata_content(full_text):

    match = re.search(
        r"\babstract\b",
        full_text,
        flags=re.IGNORECASE
    )

    if not match:

        return {
            "metadata_text": "",
            "content_text": full_text
        }

    metadata_text = full_text[:match.start()].strip()

    content_text = full_text[match.start():].strip()

    return {
        "metadata_text": metadata_text,
        "content_text": content_text
    }

STEP 3: CHUNKING CONTENT SECTION

In [ ]:
import nltk
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

STEP 3.1: Detecting sections before chunking

In [ ]:
import re

def split_into_sections(text, canonical_section_headings):

    sections = []
    current_section = "Unknown"
    current_text = []
    lines = text.split("\n")

    canonical_set_lower = {
        h.lower().strip()
        for h in canonical_section_headings
    }

    heading_patterns = [
        r"^[IVXLC]+\.\s*(.+)$",   # I. INTRODUCTION
        r"^\d+\.\s*(.+)$",        # 1. Introduction
        r"^\d+\s+(.+)$"           # 1 Introduction
    ]
    # Add a specific pattern for "Abstract —"
    abstract_heading_pattern = r"^\s*Abstract\s*—\s*(.*)$"

    for line_num, line in enumerate(lines):
        cleaned = line.strip()
        if not cleaned:
            continue

        is_new_section = False
        identified_section_name = None
        content_on_heading_line = ""

        # Attempt to match "Abstract —" first, as it's a special case and often appears at the very beginning
        abstract_match = re.match(abstract_heading_pattern, cleaned, re.IGNORECASE)
        if abstract_match and "abstract" in canonical_set_lower:
            is_new_section = True
            identified_section_name = "Abstract"
            content_on_heading_line = abstract_match.group(1).strip()
        else:
            # Check for direct heading match (e.g., "Introduction")
            if cleaned.lower() in canonical_set_lower:
                is_new_section = True
                identified_section_name = next(
                    h for h in canonical_section_headings if h.lower().strip() == cleaned.lower()
                )

            # Check for patterned headings (e.g., "1. Introduction", "I. INTRODUCTION")
            if not is_new_section:
                for pattern in heading_patterns:
                    match = re.match(pattern, cleaned, flags=re.IGNORECASE)
                    if match:
                        extracted_name = match.group(1).strip()
                        if extracted_name.lower() in canonical_set_lower:
                            is_new_section = True
                            identified_section_name = next(
                                (h for h in canonical_section_headings if h.lower().strip() == extracted_name.lower()),
                                extracted_name
                            )
                            break

        if is_new_section:
            if current_text: # If there was accumulated text for the previous section
                sections.append({
                    "section": current_section,
                    "text": "\n".join(current_text).strip() # strip the final text block
                })
            current_section = identified_section_name
            current_text = []
            if content_on_heading_line: # Add content that was on the same line as the heading (e.g., after "Abstract —")
                current_text.append(content_on_heading_line)
        else:
            current_text.append(line)

    if current_text: # Add the last accumulated text
        sections.append({
            "section": current_section,
            "text": "\n".join(current_text).strip()
        })

    return sections

In [ ]:
from nltk.tokenize import sent_tokenize

def create_chunks(
    text,
    chunk_size=1000,
    overlap_sentences=2
):

    sentences = sent_tokenize(text)

    chunks = []

    current_chunk = []

    for sentence in sentences:

        candidate = " ".join(current_chunk + [sentence])

        if len(candidate) <= chunk_size:

            current_chunk.append(sentence)

        else:

            chunks.append(" ".join(current_chunk))

            current_chunk = (
                current_chunk[-overlap_sentences:]
                + [sentence]
            )

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

STEP 4: CREATE EMBEDDINGS

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from sklearn.preprocessing import normalize



STEP 5: Using FAISS

In [ ]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 79.1 MB/s eta 0:00:00


In [ ]:
import faiss


STEP 6: Using Ollama for synthesis

In [ ]:
#defining a route query method to identify query intent

def route_query(query):

    query = query.lower()

    if any(word in query for word in
           ["objective", "aim", "purpose", "goal"]):
        return ["Abstract", "Introduction"]

    elif any(word in query for word in
             ["method", "methodology", "approach", "datasets", "materials","experimental setup"]):
        return ["Methodology"]

    elif any(word in query for word in
             ["result", "accuracy", "performance","experimental results", "validation"]):
        return ["Results and Discussion"]

    elif any(word in query for word in
             ["conclusion", "future work"]):
        return ["Conclusion"]

    return section_headings

In [ ]:
from sklearn.preprocessing import normalize


def retrieve_from_sections(
    query,
    embedding_model,
    filtered_docs,
    temp_index,
    top_k=5):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    query_embedding = normalize(
        query_embedding
    )

    scores, indices = temp_index.search(
        query_embedding.astype("float32"),
        top_k
    )

    results = []

    for score, idx in zip(
        scores[0],
        indices[0]
    ):

        results.append({

            "score": float(score),

            "section":
                filtered_docs[idx]["section"],

            "text":
                filtered_docs[idx]["text"]

        })

    return results

In [ ]:
!pip install ollama
import ollama

In [ ]:
!sudo apt-get install -y zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 3 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (7,535 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 118243 files and directories currently 

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
# Start Ollama in the background
import subprocess
import os

# Set the OLLAMA_HOST environment variable to allow connections from outside the container
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'

# Command to start Ollama server
command = "/usr/local/bin/ollama serve"

# Start the process in the background, redirecting output to /dev/null
process = subprocess.Popen(command.split(), stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("Ollama server started in the background.")

Ollama server started in the background.


In [ ]:
!ollama pull qwen2.5:1.5b

In [ ]:
import ast
import numpy as np
import faiss
from sklearn.preprocessing import normalize
import time # Import time module if not already imported

def ask_paper(query):

    # Step 1: Route query to relevant sections

    selected_sections = [
    section.strip().lower()
    for section in route_query(query)
]


    # Step 2: Filter documents

    filtered_docs = [

    doc

    for doc in documents

    if doc["section"].strip().lower()
       in selected_sections
]

# Fallback if router selected
# sections that don't exist

    if len(filtered_docs) == 0:

        print(
            "Router selected unavailable sections."
        )

        print(
            "Selected Sections:",
            selected_sections
        )

        print(
            "Using full document as fallback."
        )

        filtered_docs = documents
    # Step 3: Build temporary FAISS index

    filtered_embeddings = np.array(

        [
            doc["embedding"]
            for doc in filtered_docs
        ],

        dtype="float32"
    )

    print("Selected Sections:", selected_sections)
    print("Filtered Docs:", len(filtered_docs))
    print("Filtered Embeddings Shape:",
          filtered_embeddings.shape)


    print("Building FAISS index")
    temp_index = faiss.IndexFlatIP(
        filtered_embeddings.shape[1]
    )

    temp_index.add(
        filtered_embeddings
    )

    print("Encoding the query")

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    query_embedding = normalize(
        query_embedding
    )

    print("Query embedded, now fetching top 10 chunks")

    scores, indices = temp_index.search(
        query_embedding.astype("float32"),
        5
    )

    retrieved_context = "\n\n".join(

        filtered_docs[idx]["text"]

        for idx in indices[0]
    )


    print("Context chars:", len(retrieved_context))
    print("Context words:", len(retrieved_context.split()))

    # calling Ollama for generation

    prompt = f"""
You are a research paper assistant.

Answer the user's question using ONLY the
information provided below.

For questions about:
- title
- authors
- affiliations
- institution
- email

use the Metadata section.

For all other questions use the Retrieved Context.

If the answer is not present,
say that the information is not available.

Metadata:
{paper_parts['metadata_text']}

Retrieved Context:
{retrieved_context}

Question:
{query}



"""


    start = time.time()

    response = ollama.chat(

    model="qwen2.5:1.5b",

    messages=[

        {
            "role": "user",
            "content": prompt
        }

    ]
)

    print("Generation Time:",
      time.time() - start)


    return response["message"]["content"]


In [ ]:
from sklearn.preprocessing import normalize

#extract text from the pdf
full_extracted_text = extract_text_pymupdf("/content/mlg.pdf")

#extract headigs from the extracted text
section_headings = extract_section_headings(full_extracted_text)

#splitting into metadata and content parts
paper_parts = split_metadata_content(full_extracted_text)
content_part = paper_parts['content_text']
metadata_part = paper_parts['metadata_text']

#splitting content part into sections and chunking within each section
sections = split_into_sections(content_part, KNOWN_SECTIONS) # Changed section_headings to KNOWN_SECTIONS

print("Sections found:", len(sections))

for section in sections:
    print(section["section"])

#chunk within each section
section_chunks = []

for section in sections:

    chunks = create_chunks(
        section["text"]
    )

    for chunk in chunks:

        section_chunks.append({

            "section": section["section"],

            "text": chunk,

            "embedding_text":
                f"SECTION: {section['section']}\n\n{chunk}"
        })


Sections found: 1
Abstract


In [ ]:
chunk_texts = [
    chunk["embedding_text"]
    for chunk in section_chunks
]

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

chunk_embeddings = normalize(
    chunk_embeddings
)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
documents = []

for idx, chunk in enumerate(section_chunks):

    documents.append({

        "chunk_id": idx,

        "section": chunk["section"],

        "text": chunk["text"],

        "embedding": chunk_embeddings[idx]
    })

In [ ]:
print(ask_paper(
    "What future enhancements have been proposed?"))


Router selected unavailable sections.
Selected Sections: []
Using full document as fallback.
Selected Sections: []
Filtered Docs: 53
Filtered Embeddings Shape: (53, 384)
Building FAISS index
Encoding the query
Query embedded, now fetching top 10 chunks
Context chars: 4651
Context words: 717
Generation Time: 83.25097322463989
Future enhancements for the method's application have been proposed:

1. Generalizability: The framework can handle other Romanized Indian languages with minimal changes.
2. Repetition Reduction: This technique is applied to simplify exaggerated expressions in social media text, making it easier to handle repetitive sequences of characters.
3. Outlier Removal: Short or extremely long text samples are removed to eliminate unusual entries, helping to improve the system's reliability and consistency.
4. Tokenization: The text is split into smaller units for frequency-based feature extraction.
5. Stemming: Word variants can be reduced to their root forms to enhance gen

## Key Findings

1. Semantic embeddings successfully captured research paper content.
2. FAISS retrieval consistently returned relevant chunks.
3. Generation produced accurate answers with limited hallucination.
4. Section-aware retrieval improved performance.
5. The system remained functional even when section detection failed.
6. Metadata extraction was identified as the primary limitation for some IEEE layouts.

Future Improvements:
- Hybrid Retrieval (BM25 + Dense Retrieval)
- Improved Metadata Extraction
- Faster Local Generation Models
- Frontend Interface